# E002 training on Kaggle

Designed to run unattended top to bottom (**Save & Run All**), with Internet **on** (for `pip install` and the ImageNet encoder weights).

Setup (manual, one time):
1. Upload `outputs/kaggle_pkg/solar_k_package.zip` (built by `scripts/package_kaggle.py`) as a **private** Kaggle dataset, e.g. `solar-k-package`.
2. Create this notebook, attach that dataset and the competition dataset.
3. Turn on a GPU accelerator and Internet access.
4. If resuming from a previous notebook version's output, attach that version's output as a dataset too and set `RESUME_FROM` below to its `last.pt` path; otherwise leave it `None`.

Checkpoints, logs, OOF/test probability maps all write directly to `/kaggle/working/outputs` (the persisted working directory) -- no copy-at-the-end step needed. Never put credentials in this notebook or in the uploaded package.

In [ ]:
import os, shutil, zipfile

PKG_ZIP = '/kaggle/input/solar-k-package/solar_k_package.zip'  # adjust to the attached dataset's path
WORKDIR = '/kaggle/working/solar_k'

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)
with zipfile.ZipFile(PKG_ZIP) as zf:
    zf.extractall(WORKDIR)
os.chdir(WORKDIR)
print(os.listdir('.'))

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os

# Competition data (read-only mount) -- adjust to match the attached dataset's path.
os.environ['SOLAR_K_DATA'] = '/kaggle/input/<competition-dataset-slug>'

# Checkpoints/logs/probs go straight into the persisted working directory.
os.environ['SOLAR_K_OUTPUT'] = '/kaggle/working/outputs'

# Preprocessing cache on fast, writable local disk (the competition mount is read-only).
os.environ['SOLAR_K_CACHE'] = '/kaggle/temp/solar_k_cache'

# Set to a last.pt path (e.g. under /kaggle/input/<prior-version-dataset>/outputs/E002/checkpoints/last.pt)
# to continue a previous notebook version's run instead of starting fresh.
RESUME_FROM = None

# Stay comfortably under Kaggle's session time limit; train.py saves a
# checkpoint and exits cleanly (then still runs the OOF/test inference
# below) once this is reached, rather than getting cut off mid-step.
MAX_HOURS = 8.5

In [ ]:
import subprocess

cmd = ['python', 'scripts/train.py', '--config', 'configs/E002.yaml', '--max_hours', str(MAX_HOURS)]
if RESUME_FROM:
    cmd += ['--resume', RESUME_FROM]

# One invocation covers the whole run: training (with the preprocessing
# cache built up front and 4 DataLoader workers), then -- in the same
# process -- flip-TTA OOF (fold-0 val) and test-set probability maps saved
# as compressed uint8 PNGs. No postprocessing/threshold tuning happens
# here; that runs locally against these saved probs.
subprocess.run(cmd, check=True)